In [2]:
from dotenv import load_dotenv
from google.colab import drive
import os

drive.mount('/content/drive')
load_dotenv('/content/drive/MyDrive/llm-training-pipeline/.env')
os.chdir('/content/drive/MyDrive/llm-training-pipeline')

!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import math
import time
import os
import wandb

from model import GPTConfig, GPT

In [4]:
wandb.login(key=os.getenv("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: derek-nguyen99 (derek-nguyen99-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [16]:
# Model architecture
n_layer = 6         #number of layers in neural network. each layer is 1 full transformer block
n_head = 6          #number of attention heads in each transformer block
n_embd = 384        #number of embedding dimensions 
block_size = 256    #maximum context length for predictions
dropout = 0.0       #dropout rate for regularization
bias = False        #whether to use bias in the linear layers
vocab_size = 50304  # GPT-2 vocab rounded up to nearest multiple of 64

# Data
data_dir = 'data/tinystories'
batch_size = 32                     #number of sequences per batch
gradient_accumulation_steps = 8     # effective batch = 32 * 8 * 256 = ~65k tokens

# Training
max_iters = 1000        #maximum number of training iterations
eval_interval = 250     #evaluation interval
eval_iters = 50         #number of iterations to run during evaluation
log_interval = 10       #logging interval

# Optimizer
learning_rate = 3e-4    #initial learning rate
weight_decay = 1e-1     #weight decay for regularization
beta1 = 0.9             #exponential decay rate for the first moment estimates in Adam optimizer
beta2 = 0.95            #exponential decay rate for the second moment estimates in Adam optimizer
grad_clip = 1.0         #maximum gradient norm for gradient clipping

# LR schedule
warmup_iters = 200
min_lr = 3e-5

# System
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16

print(f"Device: {device}")
print(f"Effective batch size: {batch_size * gradient_accumulation_steps * block_size:,} tokens")

Device: cuda
Effective batch size: 65,536 tokens


In [17]:
# Get a batch of data

def get_batch(split):
    path = os.path.join(data_dir, f'{split}.bin')
    data = np.memmap(path, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y

# Test it
x, y = get_batch('train')
print(f"x shape: {x.shape}")  # should be [32, 256]
print(f"y shape: {y.shape}")  # should be [32, 256]
print(f"x dtype: {x.dtype}")
print(f"Sample tokens: {x[0, :10]}")  # first 10 tokens of first sequence

x shape: torch.Size([32, 256])
y shape: torch.Size([32, 256])
x dtype: torch.int64
Sample tokens: tensor([ 2402,   257,   640,    11,   612,   373,   257, 21757,  1310,  2576],
       device='cuda:0')


In [18]:
# Test encoding and decoding
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.decode(x[0, :10].tolist()))

 upon a time, there was a lonely little girl


In [19]:
# Initialize model
model_args = dict(
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    block_size=block_size,
    bias=bias,
    vocab_size=vocab_size,
    dropout=dropout,
)

gptconf = GPTConfig(**model_args)
model = GPT(gptconf)
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params/1e6:.2f}M")
print(f"Model device: {next(model.parameters()).device}")

number of parameters: 29.94M
Model parameters: 30.04M
Model device: cuda:0


In [20]:
# Initialize optimizer
optimizer = model.configure_optimizers(
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    betas=(beta1, beta2),
    device_type='cuda'
)

print("Optimizer initialized ✓")

num decayed parameter tensors: 26, with 30,031,872 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True
Optimizer initialized ✓


In [21]:
# Learning rate schedule with linear warmup and cosine decay
def get_lr(it):
    # Linear warmup
    if it < warmup_iters:
        return learning_rate * (it + 1) / (warmup_iters + 1)
    # Minimum LR after decay
    if it > max_iters:
        return min_lr
    # Cosine decay in between
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# Sanity check -- plot a few values
print(f"LR at iter 0:    {get_lr(0):.6f}")
print(f"LR at iter 100:  {get_lr(100):.6f}")
print(f"LR at iter 200:  {get_lr(200):.6f}  ← end of warmup")
print(f"LR at iter 500: {get_lr(500):.6f} ← halfway")
print(f"LR at iter 1000: {get_lr(1000):.6f} ← end")

LR at iter 0:    0.000001
LR at iter 100:  0.000151
LR at iter 200:  0.000300  ← end of warmup
LR at iter 500: 0.000217 ← halfway
LR at iter 1000: 0.000030 ← end


In [22]:
# Mixed precision context
ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16)
scaler = torch.amp.GradScaler('cuda')

# Evaluation function
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Eval function ready ✓")

Eval function ready ✓


In [23]:
def run_experiment(exp_name, **overrides):
    # Start with baseline config
    cfg = dict(
        n_layer=6, n_head=6, n_embd=384, block_size=256,
        dropout=0.0, bias=False, vocab_size=50304,
        batch_size=32, gradient_accumulation_steps=8,
        max_iters=2000, eval_interval=500, eval_iters=50,
        log_interval=10, learning_rate=3e-4, weight_decay=1e-1,
        beta1=0.9, beta2=0.95, grad_clip=1.0,
        warmup_iters=200, min_lr=3e-5,
    )
    # Apply overrides for this experiment
    cfg.update(overrides)

    # Init model fresh for each experiment
    gptconf = GPTConfig(
        n_layer=cfg['n_layer'], n_head=cfg['n_head'],
        n_embd=cfg['n_embd'], block_size=cfg['block_size'],
        bias=cfg['bias'], vocab_size=cfg['vocab_size'],
        dropout=cfg['dropout'],
    )
    model = GPT(gptconf).to(device)
    optimizer = model.configure_optimizers(
        cfg['weight_decay'], cfg['learning_rate'],
        (cfg['beta1'], cfg['beta2']), 'cuda'
    )
    ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16)
    scaler = torch.amp.GradScaler('cuda')

    def get_lr(it):
        if it < cfg['warmup_iters']:
            return cfg['learning_rate'] * (it + 1) / (cfg['warmup_iters'] + 1)
        if it > cfg['max_iters']:
            return cfg['min_lr']
        decay_ratio = (it - cfg['warmup_iters']) / (cfg['max_iters'] - cfg['warmup_iters'])
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return cfg['min_lr'] + coeff * (cfg['learning_rate'] - cfg['min_lr'])

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(cfg['eval_iters'])
            for k in range(cfg['eval_iters']):
                X, Y = get_batch(split, cfg['block_size'], cfg['batch_size'])
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        model.train()
        return out

    # Init wandb run
    wandb.init(
        project="llm-training-pipeline",
        name=exp_name,
        config=cfg,
        reinit=True,
    )

    iter_num = 0
    best_val_loss = float('inf')
    X, Y = get_batch('train', cfg['block_size'], cfg['batch_size'])
    t0 = time.time()

    print(f"\n{'='*50}")
    print(f"Experiment: {exp_name}")
    print(f"Overrides: {overrides}")
    print(f"{'='*50}")

    while iter_num <= cfg['max_iters']:
        lr = get_lr(iter_num)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        if iter_num % cfg['eval_interval'] == 0:
            losses = estimate_loss()
            print(f"step {iter_num}: train {losses['train']:.4f}, val {losses['val']:.4f}, lr {lr:.6f}")
            wandb.log({"iter": iter_num, "train/loss": losses['train'], "val/loss": losses['val'], "lr": lr})

            if losses['val'] < best_val_loss:
                best_val_loss = losses['val']
                print(f"  → best val loss {best_val_loss:.4f}")

        for micro_step in range(cfg['gradient_accumulation_steps']):
            with ctx:
                logits, loss = model(X, Y)
                loss = loss / cfg['gradient_accumulation_steps']
            X, Y = get_batch('train', cfg['block_size'], cfg['batch_size'])
            scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        t1 = time.time()
        dt = t1 - t0
        t0 = t1

        if iter_num % cfg['log_interval'] == 0:
            lossf = loss.item() * cfg['gradient_accumulation_steps']
            print(f"iter {iter_num}: loss {lossf:.4f}, time {dt*1000:.2f}ms")
            wandb.log({"iter": iter_num, "train/loss_step": lossf})

        iter_num += 1

    print(f"\nDone! Best val loss: {best_val_loss:.4f}")
    wandb.finish()
    return best_val_loss

In [24]:
def get_batch(split, block_size=256, batch_size=32):
    path = os.path.join(data_dir, f'{split}.bin')
    data = np.memmap(path, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y

In [25]:
# Experiment 1: Higher learning rate
run_experiment("exp1-high-lr", learning_rate=6e-4, min_lr=6e-5)

number of parameters: 29.94M
num decayed parameter tensors: 26, with 30,031,872 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True



Experiment: exp1-high-lr
Overrides: {'learning_rate': 0.0006, 'min_lr': 6e-05}
step 0: train 10.8713, val 10.8721, lr 0.000003
  → best val loss 10.8721
iter 0: loss 10.8660, time 42318.49ms
iter 10: loss 10.0138, time 1597.08ms
iter 20: loss 9.5682, time 1654.62ms
iter 30: loss 8.9512, time 1249.89ms
iter 40: loss 8.1932, time 1311.07ms
iter 50: loss 7.4030, time 1341.07ms
iter 60: loss 6.5904, time 1347.52ms
iter 70: loss 5.9047, time 1404.61ms
iter 80: loss 5.3168, time 1354.32ms
iter 90: loss 4.9562, time 1331.22ms
iter 100: loss 4.6092, time 1326.51ms
iter 110: loss 4.3453, time 1328.36ms
iter 120: loss 4.1598, time 1330.74ms
iter 130: loss 4.0641, time 1334.52ms
iter 140: loss 4.0094, time 1334.21ms
iter 150: loss 3.7686, time 1343.71ms
iter 160: loss 3.7236, time 1342.62ms
iter 170: loss 3.6826, time 1333.30ms
iter 180: loss 3.4656, time 1323.89ms
iter 190: loss 3.4823, time 1316.90ms
iter 200: loss 3.4960, time 1321.73ms
iter 210: loss 3.4211, time 1326.04ms
iter 220: loss 3.2

iter,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇██
lr,▁█▆▃▂
train/loss,█▁▁▁▁
train/loss_step,█▃▃▃▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▁▁▁▁
iter,2000
lr,6e-05
train/loss,1.74752
train/loss_step,1.68107
val/loss,1.7782


tensor(1.7782)

In [26]:
# Experiment 2: More gradient accumulation (larger effective batch)
run_experiment("exp2-large-batch", gradient_accumulation_steps=16)

number of parameters: 29.94M
num decayed parameter tensors: 26, with 30,031,872 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True



Experiment: exp2-large-batch
Overrides: {'gradient_accumulation_steps': 16}
step 0: train 10.8591, val 10.8588, lr 0.000001
  → best val loss 10.8588
iter 0: loss 10.8593, time 9134.57ms
iter 10: loss 10.2082, time 2978.20ms
iter 20: loss 9.7864, time 2590.64ms
iter 30: loss 9.3459, time 2647.48ms
iter 40: loss 8.9239, time 2791.51ms
iter 50: loss 8.4471, time 2636.27ms
iter 60: loss 7.8762, time 2694.54ms
iter 70: loss 7.2683, time 2710.42ms
iter 80: loss 6.7898, time 2657.53ms
iter 90: loss 6.1842, time 2657.22ms
iter 100: loss 5.7986, time 2672.50ms
iter 110: loss 5.3252, time 2678.69ms
iter 120: loss 4.8563, time 2681.61ms
iter 130: loss 4.7672, time 2669.94ms
iter 140: loss 4.3766, time 2687.69ms
iter 150: loss 4.3478, time 2669.52ms
iter 160: loss 4.1823, time 2633.80ms
iter 170: loss 3.9578, time 2654.12ms
iter 180: loss 3.9736, time 2678.13ms
iter 190: loss 3.8276, time 2648.36ms
iter 200: loss 3.7342, time 2646.94ms
iter 210: loss 3.5893, time 2642.89ms
iter 220: loss 3.6084,

iter,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇███
lr,▁█▆▃▂
train/loss,█▂▁▁▁
train/loss_step,██▆▆▅▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▂▁▁▁
iter,2000
lr,3e-05
train/loss,1.85345
train/loss_step,1.82552
val/loss,1.85241


tensor(1.8524)

In [27]:
# Experiment 3: Shorter context length
run_experiment("exp3-short-context", block_size=128)

number of parameters: 29.94M
num decayed parameter tensors: 26, with 29,982,720 parameters
num non-decayed parameter tensors: 13, with 4,992 parameters
using fused AdamW: True



Experiment: exp3-short-context
Overrides: {'block_size': 128}
step 0: train 10.8795, val 10.8798, lr 0.000001
  → best val loss 10.8798
iter 0: loss 10.8870, time 4146.88ms
iter 10: loss 10.2198, time 694.36ms
iter 20: loss 9.8404, time 709.63ms
iter 30: loss 9.3596, time 707.63ms
iter 40: loss 9.0404, time 704.12ms
iter 50: loss 8.5048, time 710.36ms
iter 60: loss 7.9823, time 694.88ms
iter 70: loss 7.4281, time 688.92ms
iter 80: loss 6.7458, time 687.22ms
iter 90: loss 6.2637, time 671.68ms
iter 100: loss 5.7487, time 671.95ms
iter 110: loss 5.3013, time 683.83ms
iter 120: loss 5.0246, time 686.39ms
iter 130: loss 4.8115, time 682.93ms
iter 140: loss 4.4536, time 689.92ms
iter 150: loss 4.4276, time 691.60ms
iter 160: loss 4.3267, time 685.22ms
iter 170: loss 4.1290, time 691.72ms
iter 180: loss 3.9711, time 685.00ms
iter 190: loss 3.7357, time 681.50ms
iter 200: loss 3.7099, time 678.66ms
iter 210: loss 3.8814, time 682.38ms
iter 220: loss 3.5276, time 671.76ms
iter 230: loss 3.508

iter,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇████
lr,▁█▆▃▂
train/loss,█▂▁▁▁
train/loss_step,█▇▅▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▂▁▁▁
iter,2000
lr,3e-05
train/loss,2.08839
train/loss_step,2.21491
val/loss,2.09486


tensor(2.0949)